In [1]:
import os
import sys
import time
import json
import yaml
import torch
import numpy as np
from pathlib import Path
from datetime import datetime
from torch.utils.data import DataLoader
from tqdm.notebook import tqdm 


In [2]:

# 1. --- ENVIRONMENT & PATHS ---
os.environ["OMP_NUM_THREADS"] = "128" 
os.environ["MKL_NUM_THREADS"] = "128"
os.environ["CUDA_VISIBLE_DEVICES"] = "0" 

def find_project_root(current_path, target_folder="src"):
    current_path = Path(current_path).resolve()
    for parent in [current_path] + list(current_path.parents):
        if (parent / target_folder).exists():
            return parent
    return None

PROJECT_ROOT = find_project_root(Path.cwd()) or Path("/scratch/sp7007/MoT-DAQCNN")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.layers.quantum_convolution import QuantumConv2d
from src.utils.color_conversion import rgb_to_grayscale_tensor
from src.utils.data import load_medmnist_dataset

/scratch/sp7007/nyuenv/lib/python3.12/site-packages/pennylane/__init__.py:212: PennyLaneDeprecationWarning: PennyLane v0.44 has dropped maintainence support for NumPy < 2.0.0. You have version 1.26.4 installed. Future versions of PennyLane will not work with NumPy<2.0. Please consider upgrading NumPy using `python -m pip install numpy --upgrade`. 
  warnings.warn(


In [3]:

# 2. --- CONFIG & DEBUG INIT ---
CONFIG_PATH = PROJECT_ROOT / "configs" / "breast_mnist" / "cache_generation" / "digital_zz.yml"
with open(CONFIG_PATH, "r") as f:
    config = yaml.safe_load(f)

dataset_cfg = config.get("dataset", {})
model_cfg = config.get("model", {})
data_root = PROJECT_ROOT / "data"
SELECTED_TOPOLOGIES = ["kings", "horizontal", "vertical", "ring"]

print("🔍 [DEBUG] Model Configuration:")
print(f"   > Kernel Size: {model_cfg['kernel_size']} | Stride: {model_cfg['stride']}")
print(f"   > Mode: {model_cfg.get('mode', 'trotter')} | Evolution: {model_cfg['evolution_time']}")


🔍 [DEBUG] Model Configuration:
   > Kernel Size: 3 | Stride: 3
   > Mode: trotter | Evolution: 2.5


In [4]:


# 3. --- QUANTUM INITIALIZATION ---
print("\n🏗️ [DEBUG] Initializing QuantumConv2d...")
start_init = time.time()
q_conv = QuantumConv2d(
    in_channels=1 if dataset_cfg.get('color_space') == "GRAYSCALE" else 3,
    kernel_size=model_cfg['kernel_size'],
    stride=model_cfg['stride'],
    kernel_topology_names=SELECTED_TOPOLOGIES,
    scaling_factor=model_cfg['scaling_factor'],
    evolution_time=model_cfg['evolution_time'],
    mode=model_cfg.get("mode", "trotter"),
    quantum_device="lightning.qubit", 
    interface="torch",               
    include_correlators=True,
    encoding_mode="digital",
)
q_conv.eval()
print(f"✅ [DEBUG] Initialization took {time.time() - start_init:.2f}s")
print(f"✅ [DEBUG] Output Channels: {q_conv.out_channels}")



🏗️ [DEBUG] Initializing QuantumConv2d...
✅ [DEBUG] Initialization took 0.13s
✅ [DEBUG] Output Channels: 180


In [ ]:

# 4. --- THE VERBOSE PROCESSING LOOP ---
results = {}
H_OUT, W_OUT = 9, 9 

for split in ["train", "val", "test"]:
    print(f"\n📂 [DEBUG] Loading Split: {split.upper()}")
    ds = load_medmnist_dataset(dataset_cfg['name'], split, data_root)
    loader = DataLoader(ds, batch_size=4, shuffle=False, num_workers=0) # Smaller batch for tighter debug
    
    num_samples = len(ds)
    split_features = np.zeros((num_samples, q_conv.out_channels, H_OUT, W_OUT), dtype=np.float32)
    split_labels = np.zeros(num_samples, dtype=np.int64)

    idx = 0
    print(f"🚀 Starting Loop. Total batches: {len(loader)}")
    
    pbar = tqdm(total=len(loader), desc=f"Split: {split}")
    
    for i, (images, labels) in enumerate(loader):
        # --- PHASE 1: PREPROCESSING ---
        t0 = time.time()
        with torch.no_grad():
            if images.shape[1] == 3:
                images = rgb_to_grayscale_tensor(images)
            t_pre = time.time() - t0
            
            # --- PHASE 2: THE QUANTUM CORE ---
            # If the script hangs, it is happening BETWEEN these two lines
            t_q_start = time.time()
            try:
                q_out = q_conv(images)
            except Exception as e:
                print(f"\n❌ [CRITICAL ERROR] Batch {i} failed in Quantum Layer: {e}")
                raise e
            t_q_end = time.time() - t_q_start
            
            # --- PHASE 3: STORAGE ---
            batch_curr = images.shape[0]
            split_features[idx : idx + batch_curr] = q_out.numpy()
            
            lbl = labels.numpy().squeeze()
            if lbl.ndim == 0: lbl = np.array([lbl])
            split_labels[idx : idx + batch_curr] = lbl
            
        # --- DEBUG HEARTBEAT ---
        if i == 0:
            print(f"\n📊 [BATCH 0 HEARTBEAT]")
            print(f"   > Preprocessing: {t_pre:.4f}s")
            print(f"   > Quantum Simulation: {t_q_end:.4f}s ({t_q_end/batch_curr:.4f}s per image)")
            print(f"   > Output Shape: {q_out.shape}")
        
        if i % 5 == 0 and i > 0:
            print(f"⚡ Batch {i}/{len(loader)} | Current Speed: {t_q_end:.2f}s per batch")

        idx += batch_curr
        pbar.update(1)

    results[f"{split}_features"] = split_features
    results[f"{split}_labels"] = split_labels
    pbar.close()



📂 [DEBUG] Loading Split: TRAIN
Using downloaded and verified file: /scratch/sp7007/MoT-DAQCNN/data/breastmnist.npz
🚀 Starting Loop. Total batches: 137


Split: train:   0%|          | 0/137 [00:00<?, ?it/s]


📊 [BATCH 0 HEARTBEAT]
   > Preprocessing: 0.0000s
   > Quantum Simulation: 119.8855s (29.9714s per image)
   > Output Shape: torch.Size([4, 180, 9, 9])


In [ ]:

# 5. --- FINAL SAVE ---
print("\n💾 [DEBUG] Entering Save Phase...")
topo_short = "-".join([t[:3] for t in SELECTED_TOPOLOGIES])
filename = f"{dataset_cfg['name']}__DEBUG_k{model_cfg['kernel_size']}_t{topo_short}.npz"
save_path = data_root / "quantum_datasets" / filename
save_path.parent.mkdir(parents=True, exist_ok=True)

try:
    np.savez_compressed(
        save_path,
        train_features=results["train_features"],
        train_labels=results["train_labels"],
        val_features=results["val_features"],
        val_labels=results["val_labels"],
        test_features=results["test_features"],
        test_labels=results["test_labels"],
        metadata=json.dumps({"topologies": SELECTED_TOPOLOGIES, "hardware": "128-core-cpu"})
    )
    print(f"✨ [SUCCESS] Dataset saved to {save_path}")
except Exception as e:
    print(f"❌ [SAVE ERROR] Could not write to disk: {e}")

print("🏁 Debug run finished.")